# Лабораторная работа 8. Разработка мини-проекта на основе больших данных в сфере компьютерной лингвистики и презентация результатов

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/l08.ipynb)

**Цели занятия:**

- Освоить методологию проекта по анализу данных: проблема → данные → конвейер → анализ → визуализация → выводы → презентация
- Выполнить мини-проект «Анализ отзывов о книгах на узбекском языке»: очистка, ключевые слова, словарная тональность, сравнение групп
- Оценить качество метода (accuracy, матрица ошибок) и честно описать ограничения
- Оформить результаты в отчёт и спланировать презентацию по критериям оценивания

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Методология мини-проекта

Проект по анализу данных — это не «написать код», а ответить на **вопрос** с помощью данных и убедительно показать ответ. Каждый этап оставляет **артефакт**, который войдёт в отчёт и презентацию.

| Этап | Ключевой вопрос | Артефакт |
|---|---|---|
| 1. Проблема | Что и зачем мы хотим узнать? | цель, исследовательские вопросы, гипотезы |
| 2. Данные | Откуда данные, сколько их, какого качества? | описание набора, разведочный анализ (EDA) |
| 3. Конвейер | Как из сырых текстов получить признаки? | код очистки, нормализации, токенизации |
| 4. Анализ | Какие закономерности видны? | таблицы, метрики, ключевые слова |
| 5. Визуализация | Как показать результат за 5 секунд? | 2–4 графика с выводом в заголовке |
| 6. Выводы | Что ответили на вопрос? Где границы метода? | выводы, ограничения, дальнейшие шаги |
| 7. Презентация | Как убедить слушателей? | 8–10 слайдов, 7 минут доклада |

## Учебный проект: «Анализ отзывов о книгах на узбекском языке»

**Проблема.** Интернет-магазин получает тысячи отзывов, читать их вручную невозможно. Нужно автоматически понять, чем довольны и недовольны покупатели.

**Вопросы:** 1) как распределены оценки? 2) какие слова характерны для положительных и отрицательных отзывов? 3) может ли простой словарный метод определить тональность текста? 4) различаются ли категории книг и города?

**Данные:** `data/reviews_uz.csv` — 180 отзывов: `stars` (1–5), `text` (узбекский), `book`, `category`, `city`, `date`. Метка тональности строится по звёздам: 4–5 → `pos`, 3 → `neu`, 1–2 → `neg`.

## Шаблон проекта

**Структура папки проекта**

```text
reviews_project/
├── data/            # исходные данные (не изменяем!)
├── notebooks/       # 01_eda.ipynb, 02_analysis.ipynb
├── reports/         # графики .png, report.md
├── README.md        # цель, данные, как запустить, выводы
└── requirements.txt # pandas, matplotlib, seaborn
```

## Словарный (лексиконный) анализ тональности

Самый простой и прозрачный метод: составить словари положительных (`ajoyib`, `aʼlo`, `tavsiya`) и отрицательных (`yomon`, `qimmat`, `kechikdi`) слов и посчитать **score = (число позитивных) − (число негативных)**.

```python
def sentiment(tokens):
    score = 0
    for i, w in enumerate(tokens):
        if i + 1 < len(tokens) and tokens[i + 1] == "emas":   # «yomon emas» — не «плохо»
            continue
        if w in POS:
            score += 1
        elif w in NEG:
            score -= 1
    return score
```

> **Заметка: Отрицание в узбекском**  
> Частица **emas** стоит после слова: «yomon emas» — «неплохо». Без учёта отрицания такие отзывы ошибочно попадут в негативные. Аналогично работают аффиксы отрицания глагола (`-ma-`: kelmadi, berishmadi) — их лучше заносить в словарь целыми формами.

**Оценка качества** — сравнение предсказаний с метками по звёздам:

```python
accuracy = (df["pred"] == df["label"]).mean()          # доля верных ответов
pd.crosstab(df["label"], df["pred"])                   # матрица ошибок: где метод путается
```

## Критерии оценивания мини-проекта

| Критерий | Баллы | Что оценивается |
|---|---|---|
| Постановка задачи | 10 | ясная цель, исследовательские вопросы, актуальность |
| Данные | 15 | источник, объём, качество, этика (персональные данные, лицензия) |
| Обработка и код | 25 | воспроизводимый конвейер, нормализация, отсутствие «ручных» правок |
| Анализ и метрики | 20 | корректные расчёты, оценка качества метода, интерпретация |
| Визуализация | 15 | 2–4 понятных графика: заголовки, подписи осей, вывод на слайде |
| Презентация | 15 | структура, регламент 7 минут, ответы на вопросы |
| **Итого** | **100** | 86–100 — «отлично», 71–85 — «хорошо», 55–70 — «удовлетворительно» |

## План презентации (8–10 слайдов, 7 минут)

1. **Титульный**: тема, автор, группа, руководитель.
2. **Проблема и цель**: почему это важно, исследовательские вопросы.
3. **Данные**: источник, объём, пример записи, распределение оценок.
4. **Методы**: схема конвейера (загрузка → очистка → токены → словарь тональности → метрики).
5. **Результаты** (2–3 слайда): по одному графику на слайд; заголовок слайда — это вывод («Негативные отзывы связаны с доставкой и упаковкой»), а не «График 2».
6. **Выводы и ограничения**: ответы на вопросы, размер выборки, простота метода.
7. **Дальнейшее развитие**: больше данных, машинное обучение, лемматизация.
8. **Спасибо! Вопросы.**

> **Совет: Правила слайдов**  
> Один слайд — одна мысль; не больше 6 строк текста; шрифт не меньше 24 pt; числа округлены; код на слайды не выносится (только в приложение или репозиторий).

## Задание 1. Шаг 1. Данные и разведочный анализ

Начните проект с разведочного анализа (EDA) файла `data/reviews_uz.csv`:

- `df` — таблица отзывов;
- `n_reviews` — число отзывов, `n_books` — число разных книг;
- `stars_dist` — число отзывов по оценкам 1…5 (по возрастанию звёзд);
- `mean_stars` — средняя оценка (2 знака);
- `share_pos` — доля отзывов с оценкой 4–5 (3 знака).

Выведите краткую сводку: этими числами начнётся слайд «Данные».

<details><summary>💡 Подсказка</summary>

`len(df)`, `df["book"].nunique()`, `df["stars"].value_counts().sort_index()`, `round(df["stars"].mean(), 2)`, `round((df["stars"] >= 4).mean(), 3)`.

</details>

In [ ]:
import pandas as pd

df = pd.read_csv("data/reviews_uz.csv")
print(df.shape)
print(df.head())

n_reviews = ...
n_books = ...
stars_dist = ...
mean_stars = ...
share_pos = ...

print(f"Отзывов: {n_reviews}, книг: {n_books}, средняя оценка: {mean_stars}, доля 4–5★: {share_pos}")
stars_dist

## Задание 2. Шаг 2. Очистка, метки и ключевые слова

Подготовлены `df`, функция `normalize(text)` (регистр + апострофы) и стоп-слова `stop`.

1. Напишите `tokenize(text)`: слова `re.findall(r"\w+", normalize(text))`, только буквенные.
2. Добавьте столбцы `tokens`, `n_words` (число токенов) и `label`: `"pos"` для 4–5★, `"neu"` для 3★, `"neg"` для 1–2★.
3. `top_pos` и `top_neg` — по 5 самых частых **содержательных** слов (не из `stop`, длина ≥ 3) в положительных и отрицательных отзывах: списки пар `(слово, частота)`.

Какие темы волнуют недовольных покупателей?

<details><summary>💡 Подсказка</summary>

`return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]`; метка: `df["stars"].apply(lambda s: "pos" if s >= 4 else ("neg" if s <= 2 else "neu"))`; `top_pos = top_words(df.loc[df["label"] == "pos", "tokens"])`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/reviews_uz.csv")

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

In [ ]:
def tokenize(text):
    ...

df["tokens"] = df["text"].apply(tokenize)
df["n_words"] = ...
df["label"] = ...

def top_words(tokens_series, n=5):
    c = Counter()
    for toks in tokens_series:
        c.update(w for w in toks if w not in stop and len(w) >= 3)
    return c.most_common(n)

top_pos = ...
top_neg = ...
print("Положительные:", top_pos)
print("Отрицательные:", top_neg)
df["label"].value_counts()

## Задание 3. Шаг 3. Словарная тональность и её качество

В `df` есть `tokens` и `label`. Словари `POS` и `NEG` даны в стартовом коде (их можно расширить).

1. Напишите `sentiment(tokens)`: +1 за каждое слово из `POS`, −1 за слово из `NEG`; если **следующее** слово — `emas`, текущее оценочное слово не учитывается («yomon emas» → 0).
2. `df["score"]` — оценка каждого отзыва; `df["pred"]` — `"pos"` при score > 0, `"neg"` при score < 0, иначе `"neu"`.
3. `accuracy` — доля совпадений `pred` и `label` (3 знака); `confusion` — матрица ошибок `pd.crosstab(df["label"], df["pred"])`.

Добейтесь `accuracy` не ниже 0.95. Какие отзывы метод всё ещё путает?

<details><summary>💡 Подсказка</summary>

В начале цикла: `if i + 1 < len(tokens) and tokens[i + 1] == "emas": continue`. `df["pred"] = df["score"].apply(lambda s: "pos" if s > 0 else ("neg" if s < 0 else "neu"))`; `accuracy = round((df["pred"] == df["label"]).mean(), 3)`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/reviews_uz.csv")

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

df["tokens"] = df["text"].apply(tokenize)
df["n_words"] = df["tokens"].str.len()
df["label"] = df["stars"].apply(lambda s: "pos" if s >= 4 else ("neg" if s <= 2 else "neu"))

In [ ]:
POS = {"ajoyib", "aʼlo", "chiroyli", "xushmuomala", "tez", "tavsiya", "arzon", "sifatli", "qiziqarli"}
NEG = {"yomon", "qimmat", "kechikdi", "berishmadi", "yirtilgan", "notoʻgʻri"}

def sentiment(tokens):
    score = 0
    for i, w in enumerate(tokens):
        if w in POS:
            score += 1
        elif w in NEG:
            score -= 1
    return score

df["score"] = df["tokens"].apply(sentiment)
df["pred"] = ...
accuracy = ...
confusion = ...

print("accuracy =", accuracy)
confusion

## Задание 4. Шаг 4. Сравнение категорий и городов

В `df` есть `stars`, `label`, `score`, `category`, `city`.

1. `by_cat` — таблица по категориям книг со столбцами `n` (число отзывов), `mean_stars`, `share_neg` (доля `label == "neg"`), `mean_score`; значения округлены до 2 знаков, сортировка по `mean_stars` по убыванию.
2. `by_city` — такая же таблица по городам.
3. `best_city` — город с самой высокой средней оценкой, `worst_category` — категория с самой низкой.
4. `pivot` — сводная таблица средней оценки: строки — города, столбцы — категории.

Помните: если в группе 6 отзывов, её среднее ненадёжно. Учитывайте `n` при выводах.

<details><summary>💡 Подсказка</summary>

`mean_stars=("stars", "mean")`, `mean_score=("score", "mean")`, `.sort_values("mean_stars", ascending=False)`; `best_city = by_city["mean_stars"].idxmax()`; `pivot = df.pivot_table(index="city", columns="category", values="stars", aggfunc="mean").round(2)`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/reviews_uz.csv")

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

df["tokens"] = df["text"].apply(tokenize)
df["n_words"] = df["tokens"].str.len()
df["label"] = df["stars"].apply(lambda s: "pos" if s >= 4 else ("neg" if s <= 2 else "neu"))

POS = {"ajoyib", "aʼlo", "chiroyli", "xushmuomala", "tez", "tavsiya", "arzon", "sifatli", "qiziqarli"}
NEG = {"yomon", "qimmat", "kechikdi", "berishmadi", "yirtilgan", "notoʻgʻri"}

def sentiment(tokens):
    score = 0
    for i, w in enumerate(tokens):
        if i + 1 < len(tokens) and tokens[i + 1] == "emas":
            continue
        if w in POS:
            score += 1
        elif w in NEG:
            score -= 1
    return score

df["score"] = df["tokens"].apply(sentiment)
df["pred"] = df["score"].apply(lambda s: "pos" if s > 0 else ("neg" if s < 0 else "neu"))
accuracy = round((df["pred"] == df["label"]).mean(), 3)

In [ ]:
def summarize(col):
    return (df.groupby(col)
              .agg(n=("stars", "size"),
                   mean_stars=...,
                   share_neg=("label", lambda s: (s == "neg").mean()),
                   mean_score=...)
              .round(2)
              .sort_values(...))

by_cat = summarize("category")
by_city = ...
best_city = ...
worst_category = ...
pivot = ...

print("Лучший город:", best_city, "| худшая категория:", worst_category)
by_cat

## Задание 5. Шаг 5. Графики для презентации

Подготовьте рисунок для слайдов «Результаты» — **три** графика на одной строке (`plt.subplots(1, 3)`):

1. распределение оценок 1…5 (5 столбцов);
2. средняя оценка по категориям книг — горизонтальная диаграмма `by_cat["mean_stars"]` (6 столбцов);
3. тепловая карта `pivot` (город × категория) с подписями значений.

У каждого графика — заголовок-**вывод** (например, «Детские книги оценивают выше всех»), у столбчатых — подписи осей. Сохраните рисунок в `project_charts.png`.

<details><summary>💡 Подсказка</summary>

`by_cat["mean_stars"].sort_values().plot(kind="barh", ax=axes[1])` (или `axes[1].barh(...)`); `sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", ax=axes[2])`; `fig.savefig("project_charts.png", dpi=150, bbox_inches="tight")` перед `plt.show()`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/reviews_uz.csv")

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

df["tokens"] = df["text"].apply(tokenize)
df["n_words"] = df["tokens"].str.len()
df["label"] = df["stars"].apply(lambda s: "pos" if s >= 4 else ("neg" if s <= 2 else "neu"))

POS = {"ajoyib", "aʼlo", "chiroyli", "xushmuomala", "tez", "tavsiya", "arzon", "sifatli", "qiziqarli"}
NEG = {"yomon", "qimmat", "kechikdi", "berishmadi", "yirtilgan", "notoʻgʻri"}

def sentiment(tokens):
    score = 0
    for i, w in enumerate(tokens):
        if i + 1 < len(tokens) and tokens[i + 1] == "emas":
            continue
        if w in POS:
            score += 1
        elif w in NEG:
            score -= 1
    return score

df["score"] = df["tokens"].apply(sentiment)
df["pred"] = df["score"].apply(lambda s: "pos" if s > 0 else ("neg" if s < 0 else "neu"))
accuracy = round((df["pred"] == df["label"]).mean(), 3)

def summarize(col):
    return (df.groupby(col)
              .agg(n=("stars", "size"), mean_stars=("stars", "mean"),
                   share_neg=("label", lambda s: (s == "neg").mean()), mean_score=("score", "mean"))
              .round(2).sort_values("mean_stars", ascending=False))

by_cat = summarize("category")
by_city = summarize("city")
pivot = df.pivot_table(index="city", columns="category", values="stars", aggfunc="mean").round(2)
import seaborn as sns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

stars = df["stars"].value_counts().sort_index()
axes[0].bar(stars.index, stars.values, color="goldenrod")
axes[0].set_title("Большинство отзывов — 4 и 5 звёзд")
axes[0].set_xlabel("Звёзды")
axes[0].set_ylabel("Число отзывов")

# axes[1]: by_cat["mean_stars"] — горизонтальная диаграмма
# axes[2]: sns.heatmap(pivot, annot=True, ...)

plt.tight_layout()
plt.show()

## Задание 6. Шаг 6. Отчёт по проекту

Соберите итоги проекта в словарь `report` с ключами:

| Ключ | Содержимое |
|---|---|
| `title` | название проекта (строка) |
| `problem` | проблема и цель (строка) |
| `data` | описание данных (строка) |
| `methods` | список применённых методов (не меньше 3 пунктов) |
| `metrics` | словарь: `n_reviews`, `mean_stars` (2 знака), `share_pos` (3 знака), `accuracy` |
| `findings` | список выводов (не меньше 3), числа — из переменных |
| `limitations` | список ограничений (не меньше 1) |

Затем сгенерируйте из словаря Markdown-строку `report_md`: заголовок `# title`, разделы `## ...` (не меньше 4), выводы и ограничения — пунктами списка. Выведите отчёт.

<details><summary>💡 Подсказка</summary>

Выводы собирайте f-строками: `f"Доля положительных отзывов — {share_pos:.0%}"`. Markdown: `report_md += "\n## Результаты\n" + "\n".join("- " + x for x in report["findings"])`.

</details>

In [ ]:
# Подготовительный код задания
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/reviews_uz.csv")

def normalize(text):
    """Нижний регистр и единые апострофы: oʻ/gʻ (U+02BB), остальные — ʼ (U+02BC)."""
    text = text.lower()
    for ch in ["'", "\u2018", "\u2019", "\x60", "\u02bb"]:
        text = text.replace(ch, "\u02bc")
    return re.sub("([og])\u02bc", "\\1\u02bb", text)

stop = set(open("data/stopwords_uz.txt", encoding="utf-8").read().split())

def tokenize(text):
    return [w for w in re.findall(r"\w+", normalize(text)) if w.isalpha()]

df["tokens"] = df["text"].apply(tokenize)
df["n_words"] = df["tokens"].str.len()
df["label"] = df["stars"].apply(lambda s: "pos" if s >= 4 else ("neg" if s <= 2 else "neu"))

POS = {"ajoyib", "aʼlo", "chiroyli", "xushmuomala", "tez", "tavsiya", "arzon", "sifatli", "qiziqarli"}
NEG = {"yomon", "qimmat", "kechikdi", "berishmadi", "yirtilgan", "notoʻgʻri"}

def sentiment(tokens):
    score = 0
    for i, w in enumerate(tokens):
        if i + 1 < len(tokens) and tokens[i + 1] == "emas":
            continue
        if w in POS:
            score += 1
        elif w in NEG:
            score -= 1
    return score

df["score"] = df["tokens"].apply(sentiment)
df["pred"] = df["score"].apply(lambda s: "pos" if s > 0 else ("neg" if s < 0 else "neu"))
accuracy = round((df["pred"] == df["label"]).mean(), 3)

def summarize(col):
    return (df.groupby(col)
              .agg(n=("stars", "size"), mean_stars=("stars", "mean"),
                   share_neg=("label", lambda s: (s == "neg").mean()), mean_score=("score", "mean"))
              .round(2).sort_values("mean_stars", ascending=False))

by_cat = summarize("category")
by_city = summarize("city")
pivot = df.pivot_table(index="city", columns="category", values="stars", aggfunc="mean").round(2)

In [ ]:
report = {
    "title": "Анализ отзывов о книгах на узбекском языке",
    "problem": "...",
    "data": "...",
    "methods": [],
    "metrics": {},
    "findings": [],
    "limitations": [],
}

report_md = "# " + report["title"] + "\n"
# добавьте разделы ## Проблема, ## Данные, ## Методы, ## Результаты, ## Ограничения

print(report_md)

## Задание 7. Шаг 7. Структура презентации

_Это задание выполняется в веб-тренажёре (вопрос)._

У вас 7 минут на защиту мини-проекта. Какой порядок слайдов правильный?

## Самостоятельная работа

## Самостоятельная работа: выберите тему собственного мини-проекта

Выполните проект по той же методологии (проблема → данные → конвейер → анализ → 2–4 графика → отчёт → презентация 7 минут):

1. **«Тематика узбекских новостей»** — `data/corpus_uz/`, `data/news_raw.csv` и статьи учебного сайта: очистка, ключевые слова и облака слов по категориям, сравнение лексического разнообразия.
2. **«Анализ больших объёмов данных, формируемых в социальных сетях»** — `data/tweets_stream.csv`: динамика постов во время Навруза, платформы и языки, хештеги, тональность узбекских постов.
3. **«Читательские предпочтения»** — `data/sales.xlsx` + `data/reviews_uz.csv`: связь продаж, цены и оценок книг, сезонность.
4. **«Успеваемость студентов-лингвистов»** — `data/students.csv`: факторы GPA (посещаемость, уровень английского, город, курс).
5. **«Латиница и кириллица в узбекских текстах»** — `data/corpus_cyr/` и `data/news_raw.csv`: доля текстов в каждой письменности, транслитерация, частотные различия.

Тема для эссе: «Перспективы применения технологий Big Data в компьютерной лингвистике».